# Tokens

This notebook goes with `vak`. Each section takes one idea at its smallest size, small enough to check by hand.
The matching command then runs the same functions on 50,000 stories.

Before this: `uv run vak data` picks the stories. After this: step 4 counts which token follows which.

| Section | Idea | Used in |
|---|---|---|
| 1 | Tokens: characters, then BPE pieces | `vak tokens` |

## 1. Tokens: characters, then BPE pieces

A model reads ids, not text. A **token** is the piece of text behind one id. The simplest tokenizer makes each
character a token. Its **alphabet** is every different character, sorted, and a character's id is its place in it.

In [1]:
from medha.tokenizer import character_alphabet, character_ids

sentence = "Once upon a time, there was a little girl named Lily."
alphabet = character_alphabet([sentence])
print(len(alphabet), "characters in the alphabet:", alphabet)
ids = character_ids(sentence, alphabet)
print(len(ids), "ids:", ids)

23 characters in the alphabet: [' ', ',', '.', 'L', 'O', 'a', 'c', 'd', 'e', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'w', 'y']
53 ids: [4, 14, 6, 8, 0, 20, 16, 15, 14, 0, 5, 0, 19, 11, 13, 8, 1, 0, 19, 10, 8, 17, 8, 0, 21, 5, 18, 0, 5, 0, 12, 11, 19, 19, 12, 8, 0, 9, 11, 17, 12, 0, 14, 5, 13, 8, 7, 0, 3, 11, 12, 22, 2]


**What you see**

The alphabet holds `23 characters`. The space sorts first, so its id is 0, and "O" has id 4. The sentence
becomes `53 ids`, one per character: "Once" is `[4, 14, 6, 8,` and each space is a 0.

Characters make long id lists. **BPE** (byte-pair encoding) learns longer pieces. It counts every pair of
neighbouring symbols, merges the most frequent pair into one new symbol, and repeats. The vocabulary starts with
the 256 byte values and one end token, then gains one entry per merge. Here it learns 4 merges from four words with
counts: the toy from the paper that brought BPE to language models (Sennrich, Haddow and Birch, 2016).

In [2]:
from medha.tokenizer import merges, train_bpe

words = ["low"] * 5 + ["lower"] * 2 + ["newest"] * 6 + ["widest"] * 3
toy = train_bpe(words, vocab_size=256 + 1 + 4)  # 256 bytes, the end token, and room for 4 merges
print("merges, in order:", merges(toy))
print("lowest ->", toy.encode("lowest").tokens)
print("newer  ->", toy.encode("newer").tokens)

merges, in order: ['e s', 'es t', 'l o', 'lo w']
lowest -> ['low', 'est']
newer  -> ['n', 'e', 'w', 'e', 'r']


**What you see**

The first merge is `e s`: the pair e, s appears 9 times, 6 in newest and 3 in widest. The pair s, t also
appears 9 times; the library merged e, s first. Then `es t` makes est, and `l o` and `lo w` make low.

The word lowest is not in the list, yet BPE builds it from two learned pieces: `['low', 'est']`. The word newer
stays in five pieces, `['n', 'e', 'w', 'e', 'r']`, because no merge covers its pairs yet.

Used in: `uv run vak tokens` trains the same function on 50,000 stories, with room for 1,743 merges. It saves the
tokenizer to `runs/tokenizer.json` for the next steps.

## Recap

| Idea | Remember | Used next |
|---|---|---|
| Characters | one id per character: 53 ids for the sentence | the length BPE shortens |
| BPE | merge the most frequent pair, again and again; lowest becomes low + est | `vak tokens` learns 1,743 merges; every model reads its ids |